# Integração com o Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [2]:
import os

# Roboflow

In [3]:
!pip install roboflow ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.5 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 130.8 MB/s eta 0:00:0000:01


# Imports

In [4]:
import os
import glob
import random

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from roboflow import Roboflow
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Constants

In [5]:
DATASET_ROOT_PATH = '/content/drive/My Drive/TAIL/CV/basketball-players.v2i.yolo26'

In [6]:
os.listdir(DATASET_ROOT_PATH)

['README.dataset.txt',
 'README.roboflow.txt',
 'data.yaml',
 'test',
 'valid',
 'train']

# Analisando o Dataset (O Formato YOLO)

O YOLO exige que cada imagem `.jpg` tenha um arquivo `.txt` correspondente com o mesmo nome. Dentro desse `.txt`, cada linha representa um objeto detectado, no seguinte formato:
`[class_id] [x_center] [y_center] [width] [height]`

Note que os valores estão normalizados entre 0 a 1. Este exercício é importante não para o treinamento em si, porém mais para a parte de inferência no futuro, já que os fundamentos são parecidos, mesmo que aqui sejam os labels fixos

## Visualização

In [ ]:
def plot_yolo_image(image_path, label_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    h, w, _ = img.shape
    
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            class_id = int(parts[0])
            
            # O YOLO salva como (x_center, y_center, width, height) normalizados
            x_c, y_c, bbox_w, bbox_h = map(float, parts[1:])
            
            # Desnormalize as coordenadas
            # x_c_real = 
            # y_c_real = 
            # w_real = 
            # h_real = 
            
            # Calcule o canto inferior esquerdo
            # x_min = 
            # y_min = 
            
            # Cores diferentes dependendo da classe (ex: 0 pode ser jogador, 1 juiz, 2 bola)
            color = 'red' if class_id == 0 else ('blue' if class_id == 1 else 'yellow')
            
            rect = patches.Rectangle((x_min, y_min), w_real, h_real, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

    plt.axis('off')
    plt.title("Ground Truth - Bounding Boxes", fontsize=14, fontweight='bold')
    plt.show()

# Pegando uma imagem aleatória
train_images = glob.glob(f"{DATASET_ROOT_PATH}/train/images/*.jpg")
sample_img = random.choice(train_images)
sample_label = sample_img.replace("images", "labels").replace(".jpg", ".txt")

print(f"Visualizando: {sample_img}")
plot_yolo_image(sample_img, sample_label)

# Treinamento: Arquitetura e Hiperparâmetros

A Ultralytics abstrai o nosso famoso PyTorch loop (nada de `loss.backward()` manual aqui), mas a escolha dos hiperparâmetros é vital.

YOLO é bem mais simples que o PyTorch. Ele abstrai mais que o Tensorflow. Aqui o foco é realmente na escolha dos hiperparâmetros

**Arquitetura:** Recomendo utilizarem a YOLO mais recente, a 26. Vou deixar algumas referência sobre a YOLO aqui: https://docs.ultralytics.com/pt/models/yolo26/#key-features. Mas a YOLO tem alguns tipos de modelos, de tamanhos diferentes e fica a cargo de vocês escolher o melhor. Eu recomendo irem no mais simples por ser um nivelamento
 
**Augmentations:** Assim como na tarefa passada, a parte de augmentation deixo a cargo de vocês. Tem um guia aqui com alguns agumentations do YOLO: https://docs.ultralytics.com/guides/yolo-data-augmentation/#saturation-adjustment-hsv_s e tem alguns spoilers no código de como ativar ou desativar um certo augmentation

NOTA: Não foi feito nenhum tipo de augmentation no dataset original.

### Convertendo para Dataframe
Vamos primeiro estruturar esses dados soltos do JSON em um DataFrame do Pandas limpo.

In [ ]:
# Instanciando o modelo base da Ultralytics (ele baixa os pesos pré-treinados no COCO automaticamente)
model = YOLO('yolo26n.pt') 

print("Iniciando o treinamento...")

results = model.train(
    data=f"{DATASET_ROOT_PATH }/data.yaml", 
    epochs=,                     
    imgsz=640,                     # Tamanho padrão das imagens na arquitetura YOLO
    batch=,                      
    project="/content/drive/My Drive/TAIL/CV/Treinamentos_YOLO", 
    name="basquete_exp1",
    fliplr=              # Espelhar horizontalmente        
    flipud=                    
)

## Avaliando o Modelo (Métricas de Detecção)

Na detecção de objetos, precisamos avaliar a sobreposição da nossa caixa predita com o ground truth.

Aqui vai um guia de quais métricas podem ser utilizadas: https://docs.ultralytics.com/guides/yolo-performance-metrics/#conclusion

Todas as métricas descritas no guia já foram calculadas na função .val do Ultralytics, só quero que vocês printem as corretas e as interpretem

In [ ]:
# O treinamento já gera gráficos lindos na pasta do projeto, mas podemos ver as métricas finais por aqui:
metrics = model.val(split='test')

Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 31.5±14.9 MB/s, size: 49.9 KB)
val: Scanning /content/drive/My Drive/TAIL/CV/basketball-players.v2i.yolo26/valid/labels.cache... 1953 images, 72 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1953/1953 630.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 6.7it/s 18.4s0.1s
                   all       1953      14478       0.92      0.888       0.94      0.582
Speed: 1.0ms preprocess, 3.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /content/runs/detect/val
mAP50 final: 0.940
mAP50-95 final: 0.582


# Inferência e Resultados

Aqui é só uma célula pra vocês testarem como tá o modelo

In [ ]:
# O caminho exato onde o melhor modelo foi salvo
best_model_path = "/content/drive/My Drive/TAIL/CV/Treinamentos_YOLO/basquete_exp13/weights/best.pt"

# Garantia para o Colab: verificar se o treino gerou o arquivo e carregar
if os.path.exists(best_model_path):
    modelo_teste = YOLO(best_model_path)

    # Sorteando uma imagem do conjunto de teste
    test_images = glob.glob(f"{DATASET_ROOT_PATH}/test/images/*.jpg")
    sample_test = random.choice(test_images)
    print(f"Fazendo inferência em: {sample_test}")

    # Fazendo a predição (conf=0.25 filtra detecções com menos de 25% de certeza)
    results = modelo_teste.predict(source=sample_test, conf=0.25) 

    # Plotando com a função nativa do YOLO, que já desenha as caixas e as labels preditas
    res_plotted = results[0].plot()
    
    plt.figure(figsize=(12, 12))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Predição do Modelo", fontsize=14, fontweight='bold')
    plt.show()
else:
    print("O arquivo best.pt não foi encontrado. Certifique-se de que a célula de treinamento rodou até o final.")